<div style="background-color: #1A5276; padding: 20px; border-radius: 10px; text-align: center; margin-bottom: 30px;">
    <h1 style="color: white; margin: 0;">Discipline-Specific AI Teaching Assistant</h1>
    <h2 style="color: white; margin-top: 15px;">Colab + Hugging Face edition</h2>
    <p style="color: white; margin-top: 10px; font-style: italic;">Post-seminar deep-dive — no AWS required</p>
</div>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aws-dsu/mlu-faculty-ai-seminar-lab/blob/main/discipline-assistant-colab.ipynb)

## What this is

This is the **technical deep-dive version** of the seminar's discipline-tutor lab. It does the same thing as the SageMaker version (RAG-grounded AI tutor for your course material) but uses **free, no-AWS tools**:

- **Google Colab** for the notebook environment (browser-only, no install)
- **Hugging Face Inference API** for the language model (free tier)
- **sentence-transformers** for embeddings (runs locally in Colab, no API needed)

## Quick start

1. Click the **Open in Colab** badge above
2. Sign in to Google when prompted
3. You'll need a **free Hugging Face account** — [sign up here](https://huggingface.co/join) if you don't have one
4. Generate a free [Hugging Face API token](https://huggingface.co/settings/tokens) (read access is enough)
5. Run the cells top to bottom — paste your token when prompted in cell 1.2

Total setup time: ~5 minutes. After that, the lab runs end-to-end in ~10 minutes.

## When to use this vs. the other paths

| Path | When |
|---|---|
| **This notebook (Colab + HF)** | You want to see and edit the code, post-seminar, no AWS access |
| **PartyRock template** ([repo](https://github.com/aws-dsu/mlu-faculty-ai-partyrock-template)) | You want zero code, fastest path to a working tool |
| **SageMaker + Bedrock** (`discipline-assistant.ipynb` in this same repo) | You still have AWS access from the seminar and want the original implementation |

---
# Part 1 — Setup

Install the libraries we need and connect to Hugging Face.

### 1.1 Install dependencies

Takes about 60 seconds in Colab — the sentence-transformers library is the slow part.

In [ ]:
%%capture
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pypdf huggingface_hub

### 1.2 Set your Hugging Face API token

Get one (free) at https://huggingface.co/settings/tokens — a `read` permission token is enough.

When you run this cell, paste your token when prompted. Colab hides it from view.

In [ ]:
import os
from getpass import getpass

if "HUGGINGFACEHUB_API_TOKEN" not in os.environ:
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = getpass("Paste your Hugging Face token: ")
print("Token set.")

### 1.3 Import tools and initialize the model + embeddings

We're using **Mistral 7B Instruct** for the LLM (good quality, free on HF Inference API) and **sentence-transformers** for embeddings (runs locally in Colab's CPU).

In [ ]:
import warnings
from datetime import datetime
from IPython.display import Markdown, display

from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain.prompts import PromptTemplate
from langchain.text_splitter import RecursiveCharacterTextSplitter

warnings.filterwarnings("ignore")

# LLM — free tier on HF Inference API
llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    task="text-generation",
    max_new_tokens=1500,
    temperature=0.5,
)

# Embeddings — fully local, ~90 MB download on first run
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

print("Ready.")

### 1.4 Quick sanity check — say hello

Verify the model responds.

In [ ]:
test_response = llm.invoke("Write a one-sentence greeting for a faculty AI workshop.")
print(test_response)

---
# Part 2 — Get a sample document

We'll use a public-domain CS textbook chapter as the sample document. You can swap this for your own PDF later.

### 2.1 Download a sample PDF

Pulls Persona 2's sample — Chapter 1 of Pat Morin's *Open Data Structures* (CC-BY) — from this repo's `data/` folder.

If you want to use your own PDF instead: in Colab's left sidebar, click the folder icon, then **Upload** your PDF into the working directory. Update `pdf_path` in cell 3.1 to point at it.

In [ ]:
!mkdir -p data
!curl -sL -o data/sample.pdf https://raw.githubusercontent.com/aws-dsu/mlu-faculty-ai-seminar-lab/main/data/persona2_cs_data_structures.pdf
!ls -lh data/

---
# Part 3 — Ground the AI in your document

Load the PDF, chunk it, embed the chunks, and build a search index.

This pattern is called **Retrieval-Augmented Generation (RAG)** — fancy name for *"give the model the textbook before you ask it the question."*

### 3.1 🟢 EDIT ME — Point at your PDF

In [ ]:
pdf_path = "data/sample.pdf"  # ← change to your uploaded PDF if you want

### 3.2 Load the PDF and split into searchable chunks

In [ ]:
pages = PyPDFLoader(pdf_path).load()
print(f"Loaded {len(pages)} pages from {pdf_path}")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=60,
    separators=["\n\n", "\n", "(?<=\\. )", " ", ""],
    is_separator_regex=True,
)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks.")

### 3.3 Build the searchable index

Embeddings run **locally** in Colab (no API calls). Fast — about 10 seconds for a 30-page document.

In [ ]:
print("Building index... (~10 seconds)")
vectordb = FAISS.from_documents(chunks, embeddings)
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print("Index ready.")

### 3.4 Build the grounded Q&A chain

In [ ]:
qa_template = """You are a discipline-specific teaching assistant. Use ONLY the context below to answer the question.
If the context doesn't contain the answer, say "I don't have that information in the source material." Do not make things up.

Context:
{context}

Question: {question}

Answer (cite specific details from the context):
"""

qa_chain = PromptTemplate.from_template(qa_template) | llm | StrOutputParser()

def ask(question: str):
    """Ask a question grounded in the loaded document."""
    docs = retriever.invoke(question)
    context = "\n\n".join(d.page_content for d in docs)
    answer = qa_chain.invoke({"question": question, "context": context})
    return answer, docs

print("Q&A chain ready.")

### 3.5 🟢 EDIT ME — Ask your document a question

In [ ]:
question = "What are the key concepts a student should learn from this material?"

answer, sources = ask(question)
display(Markdown(f"### Answer\n\n{answer}"))

print(f"\nAnswer drawn from {len(sources)} source chunks:")
for i, doc in enumerate(sources, 1):
    page = doc.metadata.get("page", "?")
    print(f"  [{i}] page {page}: {doc.page_content[:120]}...")

### 3.6 The grounded vs. ungrounded comparison

**This is the moment that proves grounding works.** Same question, two answers:
- **Grounded**: only the document
- **Ungrounded**: the model's general training (may invent details)

In [ ]:
comparison_question = question  # reuses your question from 3.5

grounded_answer, _ = ask(comparison_question)
vanilla_answer = llm.invoke(comparison_question)

display(Markdown(
    f"### 🟢 Grounded in YOUR document\n\n{grounded_answer}\n\n---\n\n"
    f"### 🔴 Ungrounded (model guessing from training)\n\n{vanilla_answer}"
))

> 💡 **What to look for:** The grounded answer should stick to facts from your document. The ungrounded answer may sound confident but invent details, drift off-topic, or contradict the source. **This is why grounding matters for teaching tools.**

---
# Part 4 — Generate teaching artifacts

Three templates. Pick whichever is most useful for what you teach.

### Template A — Quiz Generator

In [ ]:
quiz_request = """Based on the source material, generate 5 quiz questions for undergraduate students.
Mix 3 multiple-choice questions (with 4 options each) and 2 short-answer questions.
Include an answer key. For each question, cite which section or page of the source it tests.
Format as clean markdown."""

quiz_output, _ = ask(quiz_request)
display(Markdown(quiz_output))

### Template B — Study Guide

In [ ]:
study_guide_request = """Create a one-page study guide from the source material with three sections:

1. **Key Concepts** — 5 concepts, each with a one-sentence definition
2. **Flashcards** — 10 term/definition pairs in a table
3. **Likely Exam Questions** — 3 questions a student should be able to answer

Use clean markdown formatting."""

study_guide_output, _ = ask(study_guide_request)
display(Markdown(study_guide_output))

### Template C — Grading Rubric

In [ ]:
rubric_request = """Draft a 4-level grading rubric (Excellent / Proficient / Developing / Beginning)
for assessing student understanding of the source material.
Include 4 criteria: factual accuracy, application to new contexts, critical analysis, and use of source evidence.
Format as a markdown table."""

rubric_output, _ = ask(rubric_request)
display(Markdown(rubric_output))

### 4.4 Save your artifact to take home

In [ ]:
filename = f"my_ai_teaching_artifact_{datetime.now().strftime('%Y%m%d_%H%M')}.md"

with open(filename, "w") as f:
    f.write(f"# AI Teaching Artifacts\n\n")
    f.write(f"Source document: `{pdf_path}`\n\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}\n\n---\n\n")
    f.write(f"## Quiz\n\n{quiz_output}\n\n---\n\n")
    f.write(f"## Study Guide\n\n{study_guide_output}\n\n---\n\n")
    f.write(f"## Rubric\n\n{rubric_output}\n")

print(f"Saved to: {filename}")
print("In Colab: left sidebar → folder icon → find the file → right-click → Download.")

---
## What to try next (Monday morning)

1. **Swap the document** — upload your real syllabus, lecture notes, or textbook chapter to Colab. Re-run from Part 3.
2. **Edit the prompt templates** — match your testing style, your department's rubric criteria, your voice.
3. **Tighten the grounding** — add *"Refuse to answer if the context is unclear"* to the QA template.
4. **Try a different model** — change `repo_id` in cell 1.3. Other free HF options:
   - `meta-llama/Llama-3.2-3B-Instruct` (smaller, faster)
   - `Qwen/Qwen2.5-7B-Instruct` (different voice)
5. **Add a second document** — load multiple PDFs into one vector store for whole-course coverage.

### Where to go deeper

- The seminar's full curriculum-embedding lab: [mlu-faculty-ai-curriculum-lab](https://github.com/aws-dsu/mlu-faculty-ai-curriculum-lab)
- The no-code follow-up (PartyRock): [mlu-faculty-ai-partyrock-template](https://github.com/aws-dsu/mlu-faculty-ai-partyrock-template)
- Full MLU EEP curriculum (14 lessons, 13 labs): [aws-samples/aws-mlu-eep-generative-ai](https://github.com/aws-samples/aws-mlu-eep-generative-ai)
- See `POST_SEMINAR_LABS.md` in this repo for 17 follow-up lab ideas